# task 1
## pneumonia MRI classification

using supervised leanring we train 3 models to find pneumonia in chest MRI imagery, compare 3 models on the same pneumonia dataset and explain which one is best

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from medmnist import PneumoniaMNIST

_ = (pd, KMeans)
MAX_LEN = 3000
np.random.seed(42)


## dataset: pneumoniamnist from medmnist

- medmnist is a py lib with 122k medical images for model training, here we are using the pneumonia dataset with dual classification so either normal (healthy) or pneumonia
- the images are grayscale 28x28 chest xray images, easier than the chestmnist 3 class problem so higher scores are expected here

In [ ]:
train_dataset = PneumoniaMNIST(split='train', download=True)
val_dataset = PneumoniaMNIST(split='val')
test_dataset = PneumoniaMNIST(split='test')

X = np.concatenate([
    train_dataset.imgs,
    val_dataset.imgs,
    test_dataset.imgs,
])

y = np.concatenate([
    train_dataset.labels,
    val_dataset.labels,
    test_dataset.labels,
]).flatten()

indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

X = X[:MAX_LEN]
y = y[:MAX_LEN]

label_names = [test_dataset.info['label'][str(i)] for i in range(len(test_dataset.info['label']))]
class_names = [str(name) for name in label_names]

X_images = X.copy()
X_flat = X.reshape(X.shape[0], -1).astype(float)
X_flat = X_flat / 255.0

print('label names:', class_names)
unique, counts = np.unique(y, return_counts=True)
print('labels:', unique)
print('counts:', counts)
print('percentage:', counts / len(y) * 100)
print('dataset size after truncation:', len(y))
print('flattened feature shape:', X_flat.shape)


## data quality and preprocessing

- all images are flattened from 28x28 to 784 features
- pixel values are normalized to [0, 1] (black & white)
- same preprocessing is used for every model so the comparison stays fair

In [ ]:
unique, counts = np.unique(y, return_counts=True)
name_map = {0: class_names[0], 1: class_names[1]}
plot_names = [name_map[int(v)] for v in unique]

plt.figure(figsize=(7, 4))
bars = plt.bar(plot_names, counts, color=['steelblue', 'salmon'])
plt.title('class distribution')
plt.xlabel('class')
plt.ylabel('number of samples')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
             str(int(count)), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for row, class_value in enumerate(unique):
    class_indices = np.where(y == class_value)[0][:5]
    for col, idx in enumerate(class_indices):
        axes[row, col].imshow(X_images[idx], cmap='gray')
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name_map[int(class_value)], rotation=20, labelpad=28)
fig.suptitle('sample images by class', fontsize=11)
plt.tight_layout()
plt.show()


## experimental protocol

- same dataset, same split, same flattened features for all 3 models
- metrics: accuracy, precision, recall, f1, train time, inference time
- one required unsupervised model will use `k=5`
- kmeans with 5 clusters does **not** mean 5 disease classes: it means 5 image groups that are later mapped back to the 2 real labels


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X_flat,
    y,
    test_size=0.2,
    stratify=y,
    shuffle=True,
    random_state=42,
)

print(f'train: {len(x_train)} | test: {len(x_test)}')
results = {}


## model 1 : svc

- this is the model that already worked best in this notebook before
- it tries many `C` and `gamma` values, then keeps the best combination
- good fit for this binary task with flattened data
- main downside: slowest model here because gridsearch tests multiple combinations


In [ ]:
classifier = SVC()
parameters = [{'gamma': [0.1, 0.07, 0.05], 'C': [1, 5, 10, 15, 20]}]

t0 = time.time()
grid_search = GridSearchCV(classifier, parameters, cv=3, n_jobs=-1)
grid_search.fit(x_train, y_train)
svc_train_time = time.time() - t0

best_estimator = grid_search.best_estimator_

t0 = time.time()
y_pred_svc = best_estimator.predict(x_test)
svc_infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred_svc)
prec = precision_score(y_test, y_pred_svc, zero_division=0)
rec = recall_score(y_test, y_pred_svc, zero_division=0)
f1 = f1_score(y_test, y_pred_svc, zero_division=0)

results['SVC'] = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'train_time': svc_train_time,
    'infer_time': svc_infer_time,
}

print(f'best params: {grid_search.best_params_}')
print(f'accuracy:  {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall:    {rec:.4f}')
print(f'f1:        {f1:.4f}')
print(f'train time: {svc_train_time:.2f}s | inference time: {svc_infer_time:.4f}s')
print('\n', classification_report(y_test, y_pred_svc, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_svc, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('svc : confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.tight_layout()
plt.show()


## model 2 : random forest

- second supervised model for comparison
- much faster to train than svc
- each tree makes simple split decisions, then the forest votes
- useful baseline because it is a very different model family from svc


In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(x_train, y_train)
rf_train_time = time.time() - t0

t0 = time.time()
y_pred_rf = rf.predict(x_test)
rf_infer_time = time.time() - t0

acc = accuracy_score(y_test, y_pred_rf)
prec = precision_score(y_test, y_pred_rf, zero_division=0)
rec = recall_score(y_test, y_pred_rf, zero_division=0)
f1 = f1_score(y_test, y_pred_rf, zero_division=0)

results['RandomForest'] = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'train_time': rf_train_time,
    'infer_time': rf_infer_time,
}

print(f'accuracy:  {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall:    {rec:.4f}')
print(f'f1:        {f1:.4f}')
print(f'train time: {rf_train_time:.2f}s | inference time: {rf_infer_time:.4f}s')
print('\n', classification_report(y_test, y_pred_rf, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_rf, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title('random forest : confusion matrix')
plt.xlabel('predicted')
plt.ylabel('actual')
plt.tight_layout()
plt.show()


## model 3 : kmeans clustering (k=5) workaround
# note: this is unsurpesieved workaourn f to still use 5 clsuter idk if we need it here TODO

- this is the required unsupervised model with 5 clusters
- it does NOT mean 5 disease classes
- it means the algorithm makes 5 image groups based only on distance in feature space
- then each cluster is mapped back to `normal` or `pneumonia` by majority vote from the training set
- we use it as an analysis/comparison model, not because 5 is the natural number of labels here


In [ ]:
t0 = time.time()
km = KMeans(n_clusters=5, random_state=42, n_init=10)
km.fit(x_train)
km_train_time = time.time() - t0

train_clusters = km.predict(x_train)
cluster_to_label = {}
for cluster_id in range(5):
    mask = train_clusters == cluster_id
    if mask.sum() == 0:
        cluster_to_label[cluster_id] = 0
    else:
        majority = int(np.bincount(y_train[mask]).argmax())
        cluster_to_label[cluster_id] = majority

print('cluster to label mapping:')
for cluster_id, label in cluster_to_label.items():
    print(f'cluster {cluster_id} -> {class_names[label]} ({int((train_clusters == cluster_id).sum())} train samples)')
print(f'train time: {km_train_time:.2f}s')


In [ ]:
t0 = time.time()
test_clusters = km.predict(x_test)
km_infer_time = time.time() - t0

y_pred_km = np.array([cluster_to_label[c] for c in test_clusters])

acc = accuracy_score(y_test, y_pred_km)
prec = precision_score(y_test, y_pred_km, zero_division=0)
rec = recall_score(y_test, y_pred_km, zero_division=0)
f1 = f1_score(y_test, y_pred_km, zero_division=0)

results['KMeans(k=5)'] = {
    'accuracy': acc,
    'precision': prec,
    'recall': rec,
    'f1': f1,
    'train_time': km_train_time,
    'infer_time': km_infer_time,
}

print(f'accuracy:  {acc:.4f}')
print(f'precision: {prec:.4f}')
print(f'recall:    {rec:.4f}')
print(f'f1:        {f1:.4f}')
print(f'train time: {km_train_time:.2f}s | inference time: {km_infer_time:.4f}s')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cluster_counts = [(train_clusters == c).sum() for c in range(5)]
axes[0].bar([f'c{c}' for c in range(5)], cluster_counts, color='mediumpurple')
axes[0].set_title('kmeans cluster sizes')
axes[0].set_xlabel('cluster')
axes[0].set_ylabel('count')

cm = confusion_matrix(y_test, y_pred_km, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('kmeans : confusion matrix')
axes[1].set_xlabel('predicted')
axes[1].set_ylabel('actual')
plt.tight_layout()
plt.show()


## comparative analysis

- now all 3 models have been tested on the exact same dataset and split
- accuracy is useful, but precision / recall / f1 and speed also matter
- for medical classification, recall matters a lot because false negatives are dangerous


In [ ]:
df_results = pd.DataFrame(results).T.round(4)
df_results.index.name = 'model'
print(df_results[['accuracy', 'precision', 'recall', 'f1', 'train_time', 'infer_time']].to_string())

model_names = list(results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1']
colors = ['steelblue', 'seagreen', 'mediumpurple']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, metric in zip(axes, metrics):
    values = [results[name][metric] for name in model_names]
    bars = ax.bar(model_names, values, color=colors)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=20)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{value:.2f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
time_values = [results[name]['train_time'] for name in model_names]
bars = plt.bar(model_names, time_values, color=colors)
for bar, value in zip(bars, time_values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{value:.2f}s', ha='center', fontsize=8)
plt.title('training time comparison')
plt.ylabel('seconds')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## best model justification

- expected best model here is still svc it has slighlt better stats than random forest, if speed was a factor it would be the other way around but in this context quality < speeed
- svc won bc this is a binary problem, the data is already flattened, and tuned svc handles that setting very well
- we can note that kmeans is useful for analysis, but not usually the strongest classifier because it learns without labels


## loss function explanations

**svc : hinge loss**
`L = sum_i max(0, 1 - y_i * f(x_i))`
tries to separate classes with the largest possible margin

**random forest : gini impurity**
`G = 1 - sum_k p_k^2`
each split tries to make child nodes as pure as possible

**kmeans : inertia**
`J = sum_{c=1..5} sum_{x in c} ||x - mu_c||^2`
tries to keep each sample close to its cluster center


## comparison with external work

- the medmnist paper ([MedMNIST v2: A Large-Scale Lightweight Benchmark for 2D and 3D Biomedical Image Classification](https://arxiv.org/abs/2110.14795)) shows that medical image benchmarks are often stronger with cnn based methods than flat pixel sklearn models
- on chest xray tasks more generally, like in this paper: [CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning](https://arxiv.org/abs/1711.05225) based on the CheXNEt algo shows deep cnn models can reach very strong medical image performance (f1: 0.435(95%CI0.387,0.481) higher than the radiologist average of f1: 0.387 (95% CI 0.330, 0.442) 
- compared with that, this notebook is a simpler classical ml baseline

## conclusion

- this notebook now compares 3 models on the same pneumonia dataset: svc, random forest, and kmeans with 5 clusters
- svc should stay the reference model here because this binary task matches it well and its tuned grid is already proven to work strongly
- random forest adds a second supervised comparison and gives a speed baseline
- kmeans satisfies the unsupervised requirement and shows how far we can get with structure alone
- main limitation: flattening 28x28 xray images throws away spatial information
- best alternative: cnn
